# Querying the RADAR Knowledge Graph

This notebook demonstrates the Knowledge Graph (KG) query helpers in `radar-jupyter`.

RADAR exposes a [SPARQL](https://www.w3.org/TR/sparql11-query/) endpoint at `https://radar-service.eu/sparql` whose datasets are described with the [schema.org](https://schema.org) vocabulary. The helpers wrap that endpoint so you don't have to write SPARQL yourself.

Three query functions are available, all returning a list of `Dataset` objects:

| Function | Finds datasets by |
|---|---|
| `list_datasets_by_year(year)` | publication year (`schema:datePublished`) |
| `list_datasets_by_author(author)` | author name (`schema:creator` / `schema:Person`) |
| `list_datasets_by_institution(institution)` | publishing institution (`schema:publisher` / `schema:Organization`) |

Each `Dataset` has three fields: `id` (the full node URI), `name` (`schema:name`), and `date_published` (`schema:datePublished`). Results are ordered by publication date.

## Setup

Install the library if it isn't already available in your environment (only needed once):

In [ ]:
# %pip install radar-jupyter

In [ ]:
from radar_jupyter import (
    list_datasets_by_year,
    list_datasets_by_author,
    list_datasets_by_institution,
)

## Datasets published in a given year

Pass a four-digit year. Every dataset whose `schema:datePublished` falls in that year is returned.

In [3]:
datasets = list_datasets_by_year(2024)
print(f"{len(datasets)} datasets published in 2024")

# Show the first five
for ds in datasets[:5]:
    print(f"{ds.date_published}  {ds.name}")

310 datasets published in 2024
2024-01-02  Mpp 3.3.0
2024-01-02  Antimicrobial Prenylated Isoflavones from the Leaves of the Amazonian Medicinal Plant Vatairea guianensis Aubl.
2024-01-04  Ca2+ Pre-Intercalated Bilayered Vanadium Oxide for High-Performance Aqueous Mg-Ion Batteries
2024-01-08  Original Source Data to "Bosonic excitation spectra of superconducting Bi2Sr2CaCu2O8+\delta and YBa2Cu3O6+x extracted from scanning tunneling spectra"
2024-01-09  1H NMR Data of Linderazulene with Peaks, Integrals, Multiplets and Assignments added with TopSpin and MNova


### The `Dataset` object

Each result is a small dataclass. The `id` is the full node URI; the part after `/id/` is exactly the dataset ID that `download_and_extract()` accepts (see the last section).

In [ ]:
first = datasets[0]
print("id:            ", first.id)
print("name:          ", first.name)
print("date_published:", first.date_published)

## Datasets by author

Matching is **case-insensitive** and **substring-based**, so a fragment such as `"van de Kamp"` is enough. Author names in the KG are formatted `"Last, First"`.

In [4]:
by_author = list_datasets_by_author("van de Kamp")
print(f"{len(by_author)} datasets found")

for ds in by_author[:5]:
    print(f"{ds.date_published}  {ds.name}")

2231 datasets found
2024-10-22  Micro-CT data of Protodeuterophlebia oosterbroeki
2025-01-03  Tomographic data of SMNK-PAL 10,000a-c
2025-03-28  Micro-CT data of uropodid mites
2025-04-16  Micro-CT data of Nasonia heads
2025-04-29  Micro-CT data of histerid beetle heads


## Datasets by institution

Also a case-insensitive substring match, against the publishing organisation (`schema:publisher`). Because it is a substring match, a broad term matches many datasets — use a more specific name to narrow the results.

In [ ]:
by_institution = list_datasets_by_institution("Karlsruhe Institute of Technology")
print(f"{len(by_institution)} datasets found")

for ds in by_institution[:5]:
    print(f"{ds.date_published}  {ds.name}")

## From a query result to a download

The local part of a dataset's node URI (everything after `/id/`) is the RADAR dataset ID. That means you can feed a KG result straight into `download_and_extract()`.

In [ ]:
from radar_jupyter import download_and_extract

# Take the first dataset from one of the queries above
dataset = by_author[0]

# Extract the RADAR dataset ID from the node URI
dataset_id = dataset.id.rstrip("/").rsplit("/", 1)[-1]
print(f"Downloading {dataset.name!r} (id: {dataset_id})")

data_path = download_and_extract(dataset_id)
print(f"Dataset available at: {data_path}")

## Configuration

The SPARQL endpoint can be overridden with the `RADAR_SPARQL_URL` environment variable (for example to point at a test instance). Set it **before** importing `radar_jupyter` so the value is picked up:

| Variable | Default | Description |
|---|---|---|
| `RADAR_SPARQL_URL` | `https://radar-service.eu/sparql` | RADAR Knowledge Graph SPARQL endpoint |

In [ ]:
import os
os.environ["RADAR_SPARQL_URL"] = "https://radar-service.eu/sparql"

# Import after setting the env var so the endpoint is picked up
from radar_jupyter import list_datasets_by_year

print(len(list_datasets_by_year(2023)), "datasets published in 2023")